# Devoir Machine Learning - Stratégie de trading sur le CAC40

**Nom Prénom :**  *(à compléter)*

Prévision de la hausse du CAC40 à J+1 avec un réseau de neurones, puis backtest de la stratégie.

Installation des librairies (Colab) :

In [ ]:
%pip install -q yfinance TA-Lib tensorflow scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import talib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

np.random.seed(42)
tf.random.set_seed(42)

## 1) Téléchargement des données CAC40

In [ ]:
df = yf.download("^FCHI", start="2017-01-01", end="2022-01-01",
                 interval="1d", auto_adjust=False, progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

print(df.shape)
df.head()

## 2) Calcul des features

In [ ]:
df["var_close_j1"] = df["Close"].pct_change(1) * 100
df["var_close_j8"] = df["Close"].pct_change(8) * 100

df["ADX"] = talib.ADX(df["High"], df["Low"], df["Close"], timeperiod=20)
df["RSI"] = talib.RSI(df["Close"], timeperiod=20)
slowk, slowd = talib.STOCH(df["High"], df["Low"], df["Close"], fastk_period=14, slowk_period=3)
df["STOCH_K"] = slowk
df["CCI"] = talib.CCI(df["High"], df["Low"], df["Close"], timeperiod=20)
df["VOLAT_10"] = df["Close"].pct_change().rolling(10).std() * 100

FEATURES = ["var_close_j1", "var_close_j8", "ADX", "RSI", "STOCH_K", "CCI", "VOLAT_10"]
df[FEATURES].tail()

## 3) Construction du label

Label = 1 si hausse à J+1 et variation Close vs J-8 positive, 0 sinon.

In [ ]:
df["ret_next"] = df["Close"].pct_change().shift(-1)   # rendement du lendemain (sert aussi au backtest)
df["Label"] = ((df["ret_next"] > 0) & (df["var_close_j8"] > 0)).astype(int)

df["Label"].value_counts()

## 4) Séparation train / test et normalisation

In [ ]:
df = df.dropna().copy()

X = df[FEATURES].values
y = df["Label"].values

split = int(len(df) * 0.8)            # découpage temporel, sans mélange
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

idx_test = df.index[split:]
print("Train :", X_train_scaled.shape, "| Test :", X_test_scaled.shape)

## 5) et 6) Définition et compilation du réseau

In [ ]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(512, activation="elu"),
    Dropout(0.15),
    Dense(256, activation="relu"),
    Dropout(0.15),
    Dense(128, activation="elu"),
    Dropout(0.15),
    Dense(32, activation="relu"),
    Dropout(0.15),
    Dense(1, activation="sigmoid"),
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 7) Entraînement

In [ ]:
history = model.fit(X_train_scaled, y_train, batch_size=64, epochs=25, verbose=1)

## 8) Prédiction sur les données de test

In [ ]:
y_proba = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_proba > 0.5).astype(int)

print("Hausses prédites :", int(y_pred.sum()), "sur", len(y_pred), "jours")

## 9) Qualité de la prédiction

In [ ]:
print("Matrice de confusion :")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, zero_division=0))

## 10) Backtest de la stratégie

On achète à la clôture quand le modèle prédit une hausse, et on encaisse le rendement du lendemain. Pas de position sinon.

In [ ]:
bt = df.iloc[split:].copy()
bt["signal"] = y_pred
bt["strat_ret"] = np.where(bt["signal"] == 1, bt["ret_next"], 0.0)

n_orders = int((bt["signal"] == 1).sum())
wins = int(((bt["signal"] == 1) & (bt["ret_next"] > 0)).sum())
losses = n_orders - wins
pnl = bt["strat_ret"].sum() * 100
cum_strat = (1 + bt["strat_ret"]).prod() - 1
cum_bh = (1 + bt["ret_next"]).prod() - 1

print("Bilan de la stratégie")
print("---------------------")
print("Nombre total d'ordres :", n_orders)
print("Ordres gagnants :", wins)
print("Ordres perdants :", losses)
print("Taux de réussite : {:.1f}%".format(100 * wins / n_orders))
print("P&L : {:.2f}%".format(pnl))
print("Rendement moyen par ordre : {:.3f}%".format(pnl / n_orders))
print("Rendement cumulé stratégie : {:.2f}%".format(cum_strat * 100))
print("Rendement cumulé buy & hold : {:.2f}%".format(cum_bh * 100))

In [ ]:
cum_curve = (1 + bt["strat_ret"]).cumprod() - 1

plt.figure(figsize=(12, 6))
plt.plot(bt.index, cum_curve * 100)
plt.title("Rendement cumulé de la stratégie sur la période de test")
plt.ylabel("Rendement cumulé (%)")
plt.grid(True)
plt.show()